# Project 15A · Support Triage with Escalation (Reference solution)

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 4–5 hours |
| 📋 Prerequisites | Lessons 15.01–15.04 and 15.07; Module 10 (paired comparisons); Module 14's ticket set |
| 🔑 Keys | none offline · optional `TYPESAFE_API_KEY` (live Jev, 234 requests, < $0.003) · optional `GROQ_API_KEY` (10 live escalations) |
| 💸 Cost | $0 offline; live: well under one cent of Jev, about 5K Groq tokens |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](../README.md) |

**The job:** Orbita Cloud's support lead wants the ticket router to stop calling a large LLM on every ticket. Show, on held-out tickets, that Jev-by-default with escalation on uncertain tickets keeps the accuracy and cuts the cost, choose the escalation threshold and the outage-paging band from what mistakes cost, and write the recommendation.

This is the **reference solution**. Try the [starter notebook](../15a-project-support-triage.ipynb) first; compare afterwards (same cells, same order).

## Requirements (from the spec)

1. **Route** each ticket with Jev, escalating to a larger model when Jev's top probability is below a threshold `t`.
2. **Choose** `t` on the tune half from Orbita's costs (misroute vs escalation), and an outage page / review / ignore band from its costs.
3. **Compare** Jev only, the hybrid and the LLM on every ticket on the check half: accuracy with CIs, LLM calls, cost per 1,000 tickets, and a paired test.
4. **Report** a recommendation for the support lead.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `route_ticket(ticket, answer, t)` |
| 2 | Step 2 | `choose_threshold(tickets)` |
| 3 | Step 3 | `outage_band(tickets)` |
| 4 | Step 4 | `policy_rows(tickets, t)` |
| 5 | Step 5 | `write_report(path, evidence)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY')
LANGSMITH_PROJECT = 'm15-project-15a'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The inputs

Module 14's 234 held-out tickets in two halves (choose on **tune**, report on **check**), Jev's answers to Lesson 15.02's two questions, the escalation model's recorded labels (Module 14's few-shot `qwen3:4b`, a real recorded run), and the support lead's prices for each outcome.

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# The inputs: Module 14's 234 held-out tickets in two halves, Jev's answers, the recorded LLM labels, and the costs.
import json

from mastery.decisions import decision_cost, provenance
from mastery.decisions.orbita import lab_halves, queue_questions
from mastery.decisions.policy import Band, BandCosts, band_cost, choose_band, top_probability
from mastery.evals.stats import paired_compare, wilson_interval
from mastery.finetune.predictions import RECORDED_DIR, load_predictions
from mastery.pricing import estimate_cost

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m15"          # git-ignored
OUT_DIR.mkdir(parents=True, exist_ok=True)
TUNE, CHECK = lab_halves()                                 # choose on TUNE, report on CHECK
ANSWERS = {t["id"]: jev.invoke({"state": t["text"], "questions": queue_questions()}) for t in TUNE + CHECK}
LLM_LABEL = load_predictions(RECORDED_DIR / "fewshot-qwen3-4b.json")["labels"]   # the escalation model's recorded answers
LLM_CALL_USD = estimate_cost({"input_tokens": 400, "output_tokens": 20}, "groq:openai/gpt-oss-120b")

# ← Orbita's support lead put a price on each outcome (USD, per ticket):
MISROUTE_USD = 0.50          # an agent spends ~2 minutes re-assigning a misrouted ticket
ESCALATION_USD = 0.02        # the LLM call plus the extra seconds a customer waits, as budgeted by support
OUTAGE_COSTS = BandCosts(false_yes=20, false_no=200, review=2)   # page on-call for nothing / miss an outage / a human looks
GRID = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

print(f"tune {len(TUNE)} · check {len(CHECK)} tickets · Jev answers: {provenance(next(iter(ANSWERS.values())))}")
print(f"LLM call ≈ ${LLM_CALL_USD:.6f} · misroute ${MISROUTE_USD} · escalation ${ESCALATION_USD}")

**You should now see** `tune 117 · check 117 tickets`, where Jev's answers come from (`replay (synthetic)` without a key), and the prices.

---
## Step 2 · Route, then choose the threshold

In [ ]:
def route_ticket(ticket: dict, answer, t: float) -> dict:
    """Jev's queue if its top probability is ≥ t, else the escalation model's label. Returns id, queue, by."""
    q = answer.choices["queue"]
    if top_probability(q) >= t:
        return {"id": ticket["id"], "queue": q.choice, "by": "jev"}
    return {"id": ticket["id"], "queue": LLM_LABEL.get(ticket["id"], "error"), "by": "llm"}

Choose `t` on the tune half only.

In [ ]:
def choose_threshold(tickets: list) -> tuple:
    """The t in GRID with the lowest MISROUTE_USD × misroutes + ESCALATION_USD × escalations on `tickets`."""
    rows = []
    for t in GRID:
        routed = [route_ticket(x, ANSWERS[x["id"]], t) for x in tickets]
        mis = sum(r["queue"] != x["queue"] for r, x in zip(routed, tickets))
        esc = sum(r["by"] == "llm" for r in routed)
        rows.append({"t": t, "misroutes": mis, "escalations": esc, "cost USD": round(mis * MISROUTE_USD + esc * ESCALATION_USD, 2)})
    best = min(rows, key=lambda r: (r["cost USD"], r["escalations"]))
    return best["t"], rows


T, T_ROWS = choose_threshold(TUNE)
table(T_ROWS, ["t", "misroutes", "escalations", "cost USD"]) if T_ROWS else None
print("chosen on tune: t =", T)

**You should now see** (solution) one row per `t`: misroutes fall and escalations rise as `t` grows, and the chosen `t = 0.8` balances the two at Orbita's prices. Starter: `t = 0.5` and no table.

---
## Step 3 · Outage paging

In [ ]:
def outage_band(tickets: list) -> Band:
    """Page / review / ignore band for is_outage, chosen from OUTAGE_COSTS on `tickets` (Lesson 15.02)."""
    probs = [ANSWERS[x["id"]].nouls["is_outage"].noul for x in tickets]
    labels = [int(x["queue"] == "outage") for x in tickets]
    return choose_band(probs, labels, OUTAGE_COSTS)


BAND = outage_band(TUNE)
print("outage band:", BAND)

**You should now see** (solution) a band like `low=0.45, high=0.65`: page at or above 0.65, a human looks between 0.45 and 0.65, ignore below. Missing an outage costs 10× a false page, so the band leans towards paging.

---
## Step 4 · Three policies on the check half

In [ ]:
def policy_rows(tickets: list, t: float) -> tuple:
    """Rows for "Jev only", "hybrid @t" and "LLM only" on `tickets`, plus paired_compare(hybrid, LLM only)."""
    gold = {x["id"]: x["queue"] for x in tickets}
    ways = {"Jev only": [route_ticket(x, ANSWERS[x["id"]], 0.0) for x in tickets],
            f"hybrid @{t}": [route_ticket(x, ANSWERS[x["id"]], t) for x in tickets],
            "LLM only": [route_ticket(x, ANSWERS[x["id"]], 1.01) for x in tickets]}
    jev_usd = sum(decision_cost(ANSWERS[x["id"]]) or 0.0 for x in tickets)
    rows, right = [], {}
    for name, routed in ways.items():
        right[name] = {r["id"]: r["queue"] == gold[r["id"]] for r in routed}
        llm = sum(r["by"] == "llm" for r in routed)
        model = (0.0 if name == "LLM only" else jev_usd) + llm * LLM_CALL_USD
        mis = sum(not ok for ok in right[name].values())
        rows.append({"policy": name, "accuracy": wilson_interval(sum(right[name].values()), len(tickets)),
                     "accuracy [95% CI]": str(wilson_interval(sum(right[name].values()), len(tickets))), "n": len(tickets),
                     "LLM calls": llm, "model $ per 1k": round(model / len(tickets) * 1000, 4),
                     "total $ per 1k": round((model + mis * MISROUTE_USD + llm * (ESCALATION_USD - LLM_CALL_USD)) / len(tickets) * 1000, 2)})
    pair = paired_compare(right["LLM only"], right[f"hybrid @{t}"], names=("LLM only", f"hybrid @{t}"))
    return rows, pair


ROWS, PAIR = policy_rows(CHECK, T)
table(ROWS, ["policy", "accuracy [95% CI]", "LLM calls", "model $ per 1k", "total $ per 1k"]) if ROWS else print("(no policy table yet: TODO 4)")
print("\npaired, hybrid vs LLM only:", PAIR)

**You should now see** (solution) the hybrid matching the LLM on every ticket (about 96% for both) and above Jev only (about 92%), with 14 of 117 tickets escalated. Its model cost per 1,000 tickets is about a quarter of the LLM's, and "total" (with misroutes priced in) is what the support lead pays: the cheapest of the three. The paired test finds no detectable difference to the LLM (p = 1, 2 wins each).

In [ ]:
# 💸 Optional: re-run up to 10 of the CHECK half's escalations with Groq's fast model (llm_decider, ~10 calls, ~5K tokens).
ESCALATED = [x for x in CHECK if route_ticket(x, ANSWERS[x["id"]], T)["by"] == "llm"][:10]
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (the recorded qwen3:4b labels stand in for the escalation model).")
elif not ESCALATED:
    print("(no escalations yet: TODO 1)")
else:
    from mastery.decisions import llm_decider
    from mastery.models import get_model

    live = llm_decider(get_model("fast", temperature=0))
    hits = sum(live.invoke({"state": x["text"], "questions": queue_questions()}).choices["queue"].choice == x["queue"] for x in ESCALATED)
    print(f"live escalation: {hits}/{len(ESCALATED)} right on the escalated tickets ({settings.model_fast})")

**You should now see** `⏭️ Skipped` without a Groq key. With one, the number of escalated check tickets that Groq's fast model routes correctly. These are the hard tickets, so expect fewer right than on average.

---
## Step 5 · Report

In [ ]:
def write_report(path, evidence: dict) -> str:
    """Markdown for the support lead: Threshold, Outage alerts, Policies, Recommendation, Costs and limits."""
    hybrid = next((r for r in evidence["rows"] if r["policy"].startswith("hybrid")), None)
    llm = next((r for r in evidence["rows"] if r["policy"] == "LLM only"), None)
    b = evidence["band"]
    lines = ["# Project 15A report: support triage with escalation", "",
             "## Threshold", f"Chosen on the tune half from costs (misroute ${MISROUTE_USD}, escalation ${ESCALATION_USD}): "
             f"**t = {evidence['t']}**.", "", "| t | misroutes | escalations | cost USD |", "|---|---|---|---|",
             *[f"| {r['t']} | {r['misroutes']} | {r['escalations']} | {r['cost USD']} |" for r in evidence["t_rows"]], "",
             "## Outage alerts", f"Page on-call at p(outage) ≥ {b.high}, a human reviews {b.low}–{b.high}, ignore below {b.low} "
             f"(tune half: {b.reviewed} reviewed, {b.errors} errors, cost {b.cost}).", "",
             "## Policies", "Check half (never used for choosing):", "",
             "| policy | accuracy [95% CI] | LLM calls | model $ per 1k | total $ per 1k |", "|---|---|---|---|---|",
             *[f"| {r['policy']} | {r['accuracy [95% CI]']} | {r['LLM calls']} | {r['model $ per 1k']} | {r['total $ per 1k']} |"
               for r in evidence["rows"]], "", f"Paired (McNemar), hybrid vs LLM only: {evidence['pair']}", "",
             "## Recommendation",
             f"Deploy the **hybrid at t = {evidence['t']}**: {hybrid['LLM calls']} of {hybrid['n']} tickets reach the LLM, "
             f"accuracy {hybrid['accuracy [95% CI]']} vs {llm['accuracy [95% CI]']} for the LLM on every ticket, at "
             f"${hybrid['total $ per 1k']} vs ${llm['total $ per 1k']} per 1,000 tickets including misroutes. Re-check monthly on fresh labels.", "",
             "## Costs and limits",
             "Prices from mastery.pricing (paid tier, 2026-10-16). The escalation model is Module 14's recorded qwen3:4b; "
             "in replay mode Jev's answers are synthetic (calibrated by construction), so these numbers show the method, not Jev."]
    text = "\n".join(lines) + "\n"
    path.write_text(text, encoding="utf-8")
    return text


report = write_report(OUT_DIR / "project15a-report.md", {"t": T, "t_rows": T_ROWS, "band": BAND, "rows": ROWS, "pair": PAIR})
print(report[:1500])

---
## Step 6 · Acceptance checks

In [ ]:
def accept(name, ok):
    results[name] = bool(ok)
    print(("✅" if ok else "❌"), name)


results = {}
def ref_route(x, t):                                     # an independent router for check 2
    q = ANSWERS[x["id"]].choices["queue"]
    return (q.choice, "jev") if max(q.probabilities.values()) >= t else (LLM_LABEL.get(x["id"], "error"), "llm")
def ref_cost(t):
    routed = [ref_route(x, t) for x in TUNE]
    return round(sum(q != x["queue"] for (q, _), x in zip(routed, TUNE)) * MISROUTE_USD + sum(b == "llm" for _, b in routed) * ESCALATION_USD, 2)

accept("tune and check halves: 117 + 117 tickets, no overlap",
       len(TUNE) == len(CHECK) == 117 and not {x["id"] for x in TUNE} & {x["id"] for x in CHECK})
accept("route_ticket: Jev above the threshold, the escalation model below it",
       all((route_ticket(x, ANSWERS[x["id"]], 0.8)["queue"], route_ticket(x, ANSWERS[x["id"]], 0.8)["by"]) == ref_route(x, 0.8) for x in CHECK))
accept("threshold: the cheapest t on the TUNE half", ref_cost(T) == min(ref_cost(t) for t in GRID) and len(T_ROWS) == len(GRID))
accept("outage band chosen from the costs", (BAND.low, BAND.high) == (lambda b: (b.low, b.high))(choose_band(
    [ANSWERS[x["id"]].nouls["is_outage"].noul for x in TUNE], [int(x["queue"] == "outage") for x in TUNE], OUTAGE_COSTS)))
names = [r["policy"] for r in ROWS]
accept("three policies on the CHECK half with CIs", len(ROWS) == 3 and all(r.get("n") == len(CHECK) and "[" in r["accuracy [95% CI]"] for r in ROWS))
hyb = next((r for r in ROWS if r["policy"].startswith("hybrid")), None)
jev_only = next((r for r in ROWS if r["policy"] == "Jev only"), None)
accept("hybrid: at least Jev-only accuracy with under 25% of tickets escalated",
       hyb is not None and jev_only is not None and hyb["accuracy"].estimate >= jev_only["accuracy"].estimate and hyb["LLM calls"] < 0.25 * len(CHECK))
accept("paired test: hybrid vs LLM only", PAIR is not None and PAIR.n == len(CHECK) and 0 <= PAIR.p_value <= 1)
accept("report: five sections, measured numbers, the synthetic-data note",
       all(s in report for s in ("## Threshold", "## Outage alerts", "## Policies", "## Recommendation", "## Costs and limits"))
       and "synthetic" in report and f"t = {T}" in report)
print(f"\n{sum(results.values())}/{len(results)} acceptance checks passed")
(OUT_DIR / "project15a-acceptance.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `1/8` (the data checks come from the course; everything else needs a TODO). Results: `data/outputs/m15/project15a-acceptance.json`.

---
## 🔎 Reflect (write 3–5 sentences in your report)

- Which costs would change `t` the most, and who in a real support team should own those numbers?
- What would you monitor after deployment (escalation rate, Jev's confidence distribution, misroutes reported by agents)?

## ✅ Before you commit

- [ ] Restart & Run All passes without any key.
- [ ] All TODOs done; 8/8 acceptance checks; report written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Project 15A done.** Next: [Project 15B · Agent Safety Gate](../../../../modules/15-decision-models-with-jev/project/15b-project-agent-safety-gate.ipynb).

Back to the [Module 15 README](../../../../modules/15-decision-models-with-jev/README.md) · [Course outline](../../../../COURSE_OUTLINE.md)